In [1]:
import numpy as np
from IPython.display import Image


# FEniCSx Navier-Stokes (NS) Solver
---
Technical solver details:
- Taylor-Hood elements for velocity and pressure
- BDF2 time-stepping scheme

Goals for this solver:
- 2D Exact Taylor-Green vortex
    - Compare norms
- 3D Exact solution Steinmann vortex problem
    - Compare norms
- 2D DFG benchmark problem
    - Does solver preform as expected?
- Simple anulus spinal geometry

## Taylor-Green Vortex Problem [[1]](https://en.wikipedia.org/wiki/Taylor%E2%80%93Green_vortex)

The original work describes a 3 dimensional flow. It's defined by the three velocity components $\boldsymbol v = (u,v,w)$ at time $t = 0$ governed bt the three equations:

$$
\begin{align*}
u = A \cos(ax)\sin(by)\sin(cz), \\
v = B \sin(ax)\cos(by)\sin(cz), \\
w = C \sin(ax)\sin(by)\cos(cz). 
\end{align*}
$$

The continuity eq. $\nabla \cdot \boldsymbol v = 0$ determines that $Aa + Bb + Cc = 0$.

An exact solution is known in two spatial dimensions, and it's what we will try to replicate with the FEniCSx NS solver.

### The Incompressible NS eq.
The incompressible Navier-Stokes equations are given by:

$$
\begin{align*}
&\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0, \\
&\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} + v \frac{\partial u}{\partial y} = - \frac 1 \rho \frac{\partial p}{\partial x} + \nu \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right), \\
&\frac{\partial v}{\partial t} + u \frac{\partial v}{\partial x} + v \frac{\partial v}{\partial y} = - \frac 1 \rho \frac{\partial p}{\partial y} + \nu \left( \frac{\partial^2 v}{\partial x^2} + \frac{\partial^2 v}{\partial y^2} \right).
\end{align*}
$$

The first equation is the continuity equation. The two under are the momentum equations.


### Taylor-Green Vortex Exact Solution

In an infinite domain, the doubly periodic solution is given by 

$$
\begin{align*}
u = U_0 \sin(kx)\cos(ky)F(t), & \quad v = -U_0 \cos(kx)\sin(ky)F(t), \\
F(t)&=e^{-2\nu k^2t}
\end{align*}
$$

Here $U_0$ is the maximum velocity in the flow field, $k$ is the inverse length scale, $\nu$ is the kinematic viscosity. From Taylor and Green we know that with $A=a=b=1$ we get this exact solution. Further we can expand the exponential as a Taylor series, $F(t) = 1 - 2\nu k^2t + \mathcal O(t^2)$. The pressure field $p$ is obtained by substituting the velocity solution in to the momentum equations, 

$$
\begin{align*}
p = \frac{\rho U_0^2}{4}(\cos(2k x)+ \cos (2 k y)) F(t)^2
\end{align*}
$$

With $\rho$ being the fluid density.

The stream function of the Taylor-Green vortex is given by, that satisfies $\boldsymbol v = \nabla \times \psi$,

$$
\begin{align*}
\psi = U_0 \sin(kx)\sin(ky)F(t) \boldsymbol{\hat z}.
\end{align*}
$$

With the similar vorticity given by, that also satisfies $\boldsymbol \omega = \nabla \times \boldsymbol v$,

$$
\begin{align*}
\omega = 2U_0\sin(kx)\sin(ky)F(t)\boldsymbol{\hat z}.   
\end{align*}
$$
---

$$
\boldsymbol{u}(x,y,t) =
\begin{pmatrix}
u(x,y,t)\
v(x,y,t)
\end{pmatrix}
$$

$$

p(x,y,t) =
\frac{\rho U_0^2}{4}
\left[
\cos(2kx)+\cos(2ky)
\right]
e^{-2\nu k^2t}.
$$

$$
\frac{p}{\rho} =
\frac{U_0^2}{4}
\left[
\cos(2kx)+\cos(2ky)
\right]
e^{-2\nu k^2t}.
$$


$$
\boldsymbol{f}=\boldsymbol{0}.
$$

$$
\int_\Omega p,dx=0.
$$

In [2]:
from abc import ABC, abstractmethod
import ufl
from petsc4py import PETSc
from dolfinx import fem
from dolfinx.fem.petsc import LinearProblem

from dolfinx import mesh
from mpi4py import MPI

def t_h_space(msh):
    gdim = msh.geometry.dim
    V = fem.functionspace(msh, ("Lagrange", 2, (gdim,))) # P2
    Q = fem.functionspace(msh, ("Lagrange", 1)) # P1
    return V, Q

def DirectLU():
    return {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}


class Solver(ABC):
    def __init__(self, case, numerical_params, preconditioner=None, observers=()):
        self.case, self.numerical_params = case, numerical_params
        self.pc = preconditioner or self.default_preconditioner()
        self.observers = list(observers)

    def default_preconditioner(self):
        return DirectLU()

    def solve(self):
        self.ksp.solve(self.b, self.x)

class SteadyState(Solver):
    pass

class TimeDependence(Solver): 
    def __init__(self, case, numerical_params, scheme, observers=()):
        super().__init__(case, numerical_params, observers=observers)
        self.t0, self.dt, self.T = self.numerical_params.t0, self.numerical_params.dt, self.numerical_params.T
        self.scheme = scheme
        
    def run(self) :
        self.t = self.t0
        self.setup()
        
        for obs in self.observers: obs.on_start(self)
        num_steps = round((self.T - self.t0) / self.dt)
        for n in range(1, num_steps + 1):
            self.t = self.t0 + n * self.dt
            self.update_boundary_data(self.t)
            self.assemble_step()
            self.solve()
            for obs in self.observers: obs.on_step(self) 
            self.advance_history()

        for obs in self.observers: obs.on_end(self)

    @abstractmethod
    def setup(self):
        pass

    @abstractmethod 
    def update_boundary_data(self, t):
        pass

    def assemble_step(self):
        pass

    @abstractmethod
    def solve(self):
        pass

    @abstractmethod
    def advance_history(self):
        pass    

class Observer(ABC):

    def on_start(self, solver):
        pass

    def on_step(self, solver):
        pass

    def on_end(self, solver):
        pass

class ErrorNorm(Observer): 
    def __init__(self, exact, quad_degree=8):
        self.exact, self.quad_degree = exact, quad_degree

    def on_start(self, solver):
        dx = solver.dx(degree=self.quad_degree)

        e_u, e_p = solver.u_h - self.exact.u_ufl(solver.x_c, solver.t_c), solver.p_h - self.exact.p_ufl(solver.x_c, solver.t_c)
        self.f_uL2 = fem.form(ufl.inner(e_u, e_u) * dx)
        self.f_uH1 = fem.form(ufl.inner(ufl.grad(e_u), ufl.grad(e_u)) * dx)
        self.f_pL2 = fem.form(e_p * e_p * dx)

        self.hist = {"t": [], "u_L2": [], "u_H1": [], "p_L2": []}

    def _norm(self, solver, f) :
        return np.sqrt(solver.msh.comm.allreduce(fem.assemble_scalar(f), op=MPI.SUM))

    def on_step(self, solver) :
        self.hist["t"].append(solver.t)
        self.hist["u_L2"].append(self._norm(solver, self.f_uL2))
        self.hist["u_H1"].append(self._norm(solver, self.f_uH1))
        self.hist["p_L2"].append(self._norm(solver, self.f_pL2))

    def on_end(self, solver) : 
        print("Final error norms: u_L2 = {:.3e}, u_H1 = {:.3e}, p_L2 = {:.3e}".format(
            self.hist["u_L2"][-1], self.hist["u_H1"][-1], self.hist["p_L2"][-1]))


In [3]:
class SteadyNavierStokesSolver(SteadyState):
    pass

class NavierStokesSolver(TimeDependence):
    def _spaces(self) :
        self.V, self.Q = t_h_space(self.msh)
        self.u_h = fem.Function(self.V)
        self.u_n = fem.Function(self.V)
        self.u_nm = fem.Function(self.V)

        self.p_h = fem.Function(self.Q)

    def _fconst(self, v):
        return fem.Constant(self.msh, PETSc.ScalarType(v))

    def _constants(self) :
        self.rho_c, self.mu_c, self.dt_c, self.gamma_c = [self._fconst(v) for v in (self.case.physical_params.rho, 
                                                                              self.case.physical_params.mu, self.dt, self.numerical_params.gamma)]

    def _forms(self, ) :
        u, v = ufl.TrialFunction(self.V), ufl.TestFunction(self.V)
        p, q = ufl.TrialFunction(self.Q), ufl.TestFunction(self.Q)

        a0, a1, a2 = self.a_c
        e0, e1 = self.e_c

        u_ns = e0 * self.u_n + e1 * self.u_nm
        eps = lambda z: ufl.sym(ufl.grad(z))
        conv = lambda u_ns, u, v: ufl.inner(ufl.dot(ufl.grad(u), u_ns), v)

        a00 = (self.rho_c *a0 / self.dt_c * ufl.inner(u, v) * self.dx
                + 0.5 * self.rho_c * (conv(u_ns, u, v) - conv(u_ns, v, u)) * self.dx
                + 2 * self.mu_c * ufl.inner(eps(u), eps(v)) * self.dx
                + self.gamma_c * ufl.div(u) * ufl.div(v) * self.dx)

        a01 = - p * ufl.div(v) * self.dx
        a10 = - q * ufl.div(u) * self.dx

        L0 = -self.rho_c / self.dt_c * ufl.inner(a1 * self.u_n + a2 * self.u_nm, v) * self.dx
        L1 = ufl.ZeroBaseForm((q,))

        zero = self._fconst(0.0)
        a11 = zero * p * q * self.dx

         
        return [fem.form([[a00, a01], [a10, a11]]), fem.form([L0, L1])]

    def _create_problem(self) :
        pass

    def setup(self):
        self._switched = False

        self.msh, self.facet_tags = self.case.create_mesh()

        self._spaces() # Creates V, Q, u_h, u_n, u_nm, p_h
        self._constants() # Creates rho_c, mu_c, dt_c, gamma_c

        self.x_c, self.t_c = ufl.SpatialCoordinate(self.msh), self._fconst(self.t)
        self.dx = ufl.Measure("dx", self.msh)

        self.bcs = self.case.boundary_conditions(self.V, self.Q, self.facet_tags)

        one = fem.Constant(self.msh, PETSc.ScalarType(1.0))
        self._p_int = fem.form(self.p_h * self.dx)
        self._vol = self.msh.comm.allreduce(fem.assemble_scalar(fem.form(one * self.dx)), op=MPI.SUM)

        first = self.scheme.startup or self.scheme

        self.a_c = [fem.Constant(self.msh, PETSc.ScalarType(c)) for c in first.a]
        self.e_c = [fem.Constant(self.msh, PETSc.ScalarType(c)) for c in first.e]      

        a, L = self._forms()

        self.problem = LinearProblem(
            a, L, u=[self.u_h, self.p_h], bcs=[b.bc for b in self.bcs],
            petsc_options_prefix="ns_",
            petsc_options=self.pc,
        )

        self.u_n.interpolate(self.case.initial_condition(self.t0))
        if self.scheme.order == 2 and self.scheme.startup is None:
            self.u_nm.interpolate(self.case.initial_condition(self.t0 - self.dt))
        self.u_h.x.array[:] = self.u_n.x.array # Now u_h represent the state at t0


    def update_boundary_data(self, t):
        self.t_c.value = t
        for b in self.bcs:
            b.update(t)

    def solve(self):
        self.problem.solve()
        p_mean = self.msh.comm.allreduce(fem.assemble_scalar(self._p_int), op=MPI.SUM) / self._vol
        self.p_h.x.array[:] -= p_mean

    def advance_history(self):
        self.u_nm.x.array[:] = self.u_n.x.array
        self.u_n.x.array[:] = self.u_h.x.array
        self.u_nm.x.scatter_forward(); self.u_n.x.scatter_forward()

        if self.scheme.startup is not None and not self._switched:
            self._set_scheme(self.scheme)
            self._switched = True

    def _set_scheme(self, scheme):
        for c, val in zip(self.a_c, scheme.a):
            c.value = val

        for c, val in zip(self.e_c, scheme.e):
            c.value = val



In [4]:
class BoundaryCondition(ABC):
    @abstractmethod
    def update(self, t):
        pass

class Strong(BoundaryCondition):
    pass

class Weak(BoundaryCondition):
    @abstractmethod
    def bilinear_form(self, V, t):
        pass
    @abstractmethod
    def linear_form(self, V, t):
        pass


class DirichletVelocity(Strong):
    def __init__(self, V, facet_tags, marker, u_of_t):
        self.g, self.u_of_t = fem.Function(V), u_of_t
        fdim = V.mesh.topology.dim - 1
        dofs = fem.locate_dofs_topological(V, fdim, facet_tags.find(marker))
        self.bc = fem.dirichletbc(self.g, dofs)

    def update(self, t):
        self.g.interpolate(self.u_of_t(t))

class PressurePin(Strong):
    def __init__(self, Q, point, p_of_t):
        self.g, self.p_of_t = fem.Function(Q), p_of_t
        dofs = fem.locate_dofs_geometrical(Q, lambda x: np.isclose(x[0], point[0]) & np.isclose(x[1], point[1]))
        self.bc = fem.dirichletbc(self.g, dofs)

    def update(self, t):
        self.g.interpolate(self.p_of_t(t))

In [5]:


def make_TG_mesh(N, L):
    msh = mesh.create_rectangle(MPI.COMM_WORLD, [np.array([0, 0]), np.array([L, L])], [N, N], mesh.CellType.quadrilateral)
    fdim = msh.topology.dim - 1

    # Tags boundary facet with marker 1, using locate_entities_boundary and meshtags
    def boundary(x):
        return np.isclose(x[0], 0) | np.isclose(x[0], L) | np.isclose(x[1], 0) | np.isclose(x[1], L)
    
    facet_indices = mesh.locate_entities_boundary(msh, fdim, boundary)
    facet_tags = mesh.meshtags(msh, fdim, facet_indices, 1)

    return msh, facet_tags

class TaylorGreenExact:
    def __init__(self, physical_params):
        self.U0, self.nu, self.k, self.rho = physical_params.U0, physical_params.nu, physical_params.wave_number, physical_params.rho

    def _F(self, t, m) :
        return m.exp(-2*self.nu*self.k**2*t)

    def _u(self, x, t, m) :
        F = self._F(t, m)
        return (self.U0 * m.sin(self.k*x[0]) * m.cos(self.k*x[1]) * F, 
                            -self.U0 * m.cos(self.k*x[0]) * m.sin(self.k*x[1]) * F)

    def _p(self, x, t, m) :
        F = self._F(t, m)
        return self.rho * self.U0**2 / 4 * (m.cos(2*self.k*x[0]) + m.cos(2*self.k*x[1])) * F**2

    def u_np(self, t) : return lambda x : np.vstack(self._u(x, t, np))
    def p_np(self, t) : return lambda x : self._p(x, t, np)
    def u_ufl(self, x, t) : return ufl.as_vector(self._u(x, t, ufl))
    def p_ufl(self, x, t) : return self._p(x, t, ufl)

class Case(ABC):
    def __init__(self, physical_params):
        self.physical_params = physical_params

    @abstractmethod
    def create_mesh(self):  pass           # returns (mesh, facet_tags)

    @abstractmethod
    def boundary_conditions(self, V, Q, facet_tags): pass

    def body_force(self, x, t):  return None
    def initial_condition(self, t): pass    # callable for interpolation
    pressure_nullspace: bool = False

class TaylorGreenCase(Case):
    pressure_nullspace = False

    def __init__(self, N, physical_params):
        super().__init__(physical_params)
        self.N, self.L = N, 2 * np.pi / self.physical_params.wave_number 
        self.exact = TaylorGreenExact(self.physical_params)

    def create_mesh(self):
        return make_TG_mesh(self.N, self.L) 

    def boundary_conditions(self, V, Q, facet_tags):
        return [DirichletVelocity(V, facet_tags, marker=1, u_of_t=self.exact.u_np), PressurePin(Q, point=(0, 0), p_of_t=self.exact.p_np)]

    def initial_condition(self, t):
        return self.exact.u_np(t)



In [6]:
from dataclasses import dataclass

@dataclass(frozen=True)
class NumericalParameters:
        t0: float
        T: float
        dt: float
        gamma: float = 0.0

@dataclass(frozen=True)
class PhysicalParameters:
        nu: float
        rho: float
        wave_number: float
        U0: float

        @property
        def mu(self):
            return self.nu * self.rho

@dataclass(frozen=True)
class TimeScheme:
       name : str
       order : int
       a: tuple[float, float, float]
       e: tuple[float, float]
       startup: "TimeScheme | None" = None

       @classmethod
       def bdf1(cls) :
              return cls("BDF1", 1, (1.0, -1.0, 0.0), (1.0, 0.0))

       @classmethod
       def bdf2(cls, startup=True):
              return cls("BDF2", 2, (1.5, -2.0, 0.5), (2.0, -1.0),
                        startup=cls.bdf1() if startup else None)

In [7]:
phys_params = PhysicalParameters(nu=0.01, rho=1.0, U0=1.0, wave_number=2.0*np.pi)
num_params = NumericalParameters(t0=0.0, T=1.0, dt=0.01)

bdf1 = TimeScheme.bdf1()

tg = TaylorGreenCase(N=32, physical_params=phys_params)

ns = NavierStokesSolver(case=tg, numerical_params=num_params, scheme=bdf1, observers=[ErrorNorm(tg.exact)])

ns.run()

Final error norms: u_L2 = 6.863e-04, u_H1 = 9.671e-03, p_L2 = 7.075e-04
